In [7]:
from transformers import M2M100Tokenizer
from optimum.onnxruntime import ORTModelForSeq2SeqLM
import torch

In [8]:
# 로컬 모델 경로
model_path = "assets/m2m100"

# 토크나이저 & 모델 로드 (로컬 전용)
tokenizer = M2M100Tokenizer.from_pretrained(model_path, local_files_only=True)
model = ORTModelForSeq2SeqLM.from_pretrained(model_path, local_files_only=True)

def translate(text, src_lang, tgt_lang):
    tokenizer.src_lang = src_lang
    encoded = tokenizer(text, return_tensors="pt")
    forced_bos_token_id = tokenizer.get_lang_id(tgt_lang)

    with torch.no_grad():
        generated_tokens = model.generate(**encoded, forced_bos_token_id=forced_bos_token_id)

    return tokenizer.decode(generated_tokens[0], skip_special_tokens=True)

Could not find any ONNX files with standard file name decoder_model_merged.onnx, files found: [WindowsPath('decoder_model.onnx'), WindowsPath('decoder_with_past_model.onnx'), WindowsPath('encoder_model.onnx')]. Please make sure to pass a `file_name` and/or `subfolder` argument to `from_pretrained` when loading an ONNX file with non-standard file names.


In [9]:
# 번역 테스트
print("영어")
ko_to_en = translate("안녕하세요, 오늘 날씨가 참 좋네요.", "ko", "en")
en_to_ko = translate("Hello, the weather is really nice today.", "en", "ko")
print("🇰🇷 → 🇺🇸:", ko_to_en)
print("🇺🇸 → 🇰🇷:", en_to_ko)

print("\n일본어")
ko_to_ja = translate("안녕하세요, 오늘 날씨가 참 좋네요.", "ko", "ja")
ja_to_ko = translate("こんにちは、今日はとてもいい天気ですね。", "ja", "ko")
print("🇰🇷 → 🇯🇵:", ko_to_ja)
print("🇯🇵 → 🇰🇷:", ja_to_ko)

print("\n스페인어")
ko_to_es = translate("안녕하세요, 오늘 날씨가 참 좋네요.", "ko", "es")
es_to_ko = translate("Hola, el clima está muy agradable hoy.", "es", "ko")
print("🇰🇷 → 🇪🇸:", ko_to_es)
print("🇪🇸 → 🇰🇷:", es_to_ko)

print("\n러시아어")
ko_to_ru = translate("안녕하세요, 오늘 날씨가 참 좋네요.", "ko", "ru")
ru_to_ko = translate("Здравствуйте, сегодня очень хорошая погода.", "ru", "ko")
print("🇰🇷 → 🇷🇺:", ko_to_ru)
print("🇷🇺 → 🇰🇷:", ru_to_ko)

print("\n중국어 - 중국대륙")
ko_to_zh = translate("안녕하세요, 오늘 날씨가 참 좋네요.", "ko", "zh")
zh_to_ko = translate("你好，今天天气真好。", "zh", "ko")
print("🇰🇷 → 🇨🇳(간체):", ko_to_zh)
print("🇨🇳(간체) → 🇰🇷:", zh_to_ko)

영어
🇰🇷 → 🇺🇸: Hello, the weather is great today.
🇺🇸 → 🇰🇷: 안녕하세요, 오늘 날씨는 정말 좋습니다.

일본어
🇰🇷 → 🇯🇵: こんにちは、今日の天気は素晴らしい。
🇯🇵 → 🇰🇷: 안녕하세요, 오늘은 아주 좋은 날씨입니다.

스페인어
🇰🇷 → 🇪🇸: Hola, hoy el tiempo es bueno.
🇪🇸 → 🇰🇷: 안녕하세요, 오늘 날씨는 매우 좋습니다.

러시아어
🇰🇷 → 🇷🇺: Здравствуйте, сегодня погода прекрасна.
🇷🇺 → 🇰🇷: 안녕하세요, 오늘은 아주 좋은 날씨입니다.

중국어 - 중국대륙
🇰🇷 → 🇨🇳(간체): 你好,今天天气很好。
🇨🇳(간체) → 🇰🇷: 안녕하세요, 오늘 날씨가 좋습니다.


In [10]:
from opencc import OpenCC

def translate_with_traditional(text, src, dest, input_is_traditional=False, output_traditional=False):
    cc_t2s = OpenCC('t2s')  # Traditional → Simplified
    cc_s2t = OpenCC('s2t')  # Simplified → Traditional

    # 입력이 번체라면 간체로 변환
    if input_is_traditional and src == "zh":
        text = cc_t2s.convert(text)

    # 번역 모델 호출 (예시)
    translated = translate(text, src, dest)

    # 출력이 번체라면 간체 결과를 번체로 변환
    if output_traditional and dest == "zh":
        translated = cc_s2t.convert(translated)

    return translated

In [11]:
print("중국어 - 대만, 홍콩, 마카오")
zh_traditional_to_ko = translate_with_traditional("你好，今天天氣真好。", "zh", "ko", input_is_traditional=True)
ko_to_zh_traditional = translate_with_traditional("안녕하세요, 오늘 날씨가 참 좋네요.", "ko", "zh", output_traditional=True)
print("🇨🇳(번체) → 🇰🇷:", zh_traditional_to_ko)
print("🇰🇷 → 🇨🇳(번체):", ko_to_zh_traditional)

중국어 - 대만, 홍콩, 마카오
🇨🇳(번체) → 🇰🇷: 안녕하세요, 오늘 날씨가 좋습니다.
🇰🇷 → 🇨🇳(번체): 你好,今天天氣很好。
